# AI vs Real Face Detector — Dataset Preparation (Run 2)

**Run 2:** FFHQ real + StyleGAN2 fake + diffusion-generated fake faces.

This notebook prepares the exact directory structure required by the new `full_hybrid` training pipeline:

```text
data/
├── train/
│   ├── real/
│   └── fake/
├── val/
│   ├── real/
│   └── fake/
└── test/
    ├── real/
    └── fake/
```

**Split:** 70% train / 15% validation / 15% test.

**Important:** images are shuffled with a fixed seed and copied into mutually exclusive splits. Do not manually add images to multiple splits.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, random, glob

PROJECT_DIR = Path('/content/drive/MyDrive/ai-vs-real-face-detector')
DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / '_raw'

for p in [
    RAW_DIR / 'real',
    RAW_DIR / 'fake' / 'stylegan2',
    RAW_DIR / 'fake' / 'diffusion',
]:
    p.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)

## 2. Install dataset-download dependencies

Kaggle is used for FFHQ and StyleGAN2. Hugging Face Hub is used for the diffusion dataset referenced in the original notebook.

**Do not commit or copy `kaggle.json` into the project repository.**

In [ ]:
!pip -q install -U kaggle huggingface_hub

## 3. Kaggle authentication

In [ ]:
from google.colab import files
import os

KAGGLE_PATH = Path('/root/.kaggle/kaggle.json')

if not KAGGLE_PATH.exists():
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    KAGGLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    for name in uploaded:
        shutil.move(name, KAGGLE_PATH)
    os.chmod(KAGGLE_PATH, 0o600)

print("Kaggle credentials configured for this Colab session.")

## 4. Download FFHQ real faces

Using the same FFHQ source as the original notebook. We target 5,000 real images.

The raw files remain outside the train/val/test folders until splitting is complete.

In [ ]:
N_REAL = 5000

FFHQ_DL = Path('/content/ffhq_dl')
if not (FFHQ_DL / 'download_complete.txt').exists():
    !rm -rf /content/ffhq_dl
    !mkdir -p /content/ffhq_dl
    !kaggle datasets download -d pankymathur/ffhq-224k -p /content/ffhq_dl --unzip
    (FFHQ_DL / 'download_complete.txt').touch()

real_candidates = sorted(
    glob.glob('/content/ffhq_dl/**/*.png', recursive=True) +
    glob.glob('/content/ffhq_dl/**/*.jpg', recursive=True) +
    glob.glob('/content/ffhq_dl/**/*.jpeg', recursive=True)
)

print("FFHQ files found:", len(real_candidates))
assert len(real_candidates) >= N_REAL, "Not enough FFHQ images."

random.seed(42)
random.shuffle(real_candidates)

for src in real_candidates[:N_REAL]:
    dst = RAW_DIR / 'real' / Path(src).name
    if not dst.exists():
        shutil.copy2(src, dst)

print("Real images prepared:", len(list((RAW_DIR/'real').iterdir())))

## 5. Download StyleGAN2 fake faces

Use the same StyleGAN2 dataset as the original notebook.

For Run 2, we target **2,500 StyleGAN2 images** and **2,500 diffusion images**, giving 5,000 fake images total and 5,000 real images.

In [ ]:
N_STYLEGAN2 = 2500

STYLEGAN_DL = Path('/content/fake_dl')
if not (STYLEGAN_DL / 'download_complete.txt').exists():
    !rm -rf /content/fake_dl
    !mkdir -p /content/fake_dl
    !kaggle datasets download -d hyperclaw79/fakefaces -p /content/fake_dl --unzip
    (STYLEGAN_DL / 'download_complete.txt').touch()

stylegan_candidates = sorted(
    glob.glob('/content/fake_dl/**/*.png', recursive=True) +
    glob.glob('/content/fake_dl/**/*.jpg', recursive=True) +
    glob.glob('/content/fake_dl/**/*.jpeg', recursive=True)
)

print("StyleGAN2 files found:", len(stylegan_candidates))
assert len(stylegan_candidates) >= N_STYLEGAN2, "Not enough StyleGAN2 images."

random.seed(42)
random.shuffle(stylegan_candidates)

for src in stylegan_candidates[:N_STYLEGAN2]:
    dst = RAW_DIR / 'fake' / 'stylegan2' / Path(src).name
    if not dst.exists():
        shutil.copy2(src, dst)

print("StyleGAN2 prepared:", len(list((RAW_DIR/'fake'/'stylegan2').iterdir())))

## 6. Download diffusion-generated faces

The original notebook specified:

`Purdue-M2/AI-Face-FairnessBench`

as the diffusion source.

This cell downloads the dataset repository and automatically searches it for image files. Because the repository can contain multiple generators/subfolders, the files are flattened into the local `diffusion` raw folder with collision-safe names.

Target: **2,500 diffusion images**.

If the repository is too large for the Colab session, download/copy a 2,500-image diffusion subset into `RAW_DIR/fake/diffusion` manually and rerun the validation cell.

In [ ]:
from huggingface_hub import snapshot_download

N_DIFFUSION = 2500
HF_REPO = "Purdue-M2/AI-Face-FairnessBench"
HF_LOCAL = Path('/content/AI-Face-FairnessBench')

if not HF_LOCAL.exists():
    snapshot_download(
        repo_id=HF_REPO,
        repo_type='dataset',
        local_dir=str(HF_LOCAL),
        local_dir_use_symlinks=False
    )

diffusion_candidates = sorted(
    [
        str(p) for p in HF_LOCAL.rglob('*')
        if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
    ]
)

print("Diffusion image files found:", len(diffusion_candidates))
assert len(diffusion_candidates) >= N_DIFFUSION, (
    f"Only {len(diffusion_candidates)} images found; need {N_DIFFUSION}. "
    "Inspect the downloaded dataset or reduce N_DIFFUSION."
)

random.seed(42)
random.shuffle(diffusion_candidates)

for idx, src in enumerate(diffusion_candidates[:N_DIFFUSION]):
    src_path = Path(src)
    # Prefix avoids filename collisions between generator subfolders.
    dst = RAW_DIR / 'fake' / 'diffusion' / f"diffusion_{idx:06d}{src_path.suffix.lower()}"
    if not dst.exists():
        shutil.copy2(src_path, dst)

print("Diffusion prepared:", len(list((RAW_DIR/'fake'/'diffusion').iterdir())))

## 7. Raw dataset sanity check

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def image_files(folder):
    return [
        p for p in Path(folder).rglob('*')
        if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}
    ]

counts = {
    "real": len(image_files(RAW_DIR/'real')),
    "stylegan2": len(image_files(RAW_DIR/'fake'/'stylegan2')),
    "diffusion": len(image_files(RAW_DIR/'fake'/'diffusion')),
}
print(counts)

assert counts["real"] >= 1
assert counts["stylegan2"] >= 1
assert counts["diffusion"] >= 1

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
groups = [
    ("real", image_files(RAW_DIR/'real')),
    ("stylegan2", image_files(RAW_DIR/'fake'/'stylegan2')),
    ("diffusion", image_files(RAW_DIR/'fake'/'diffusion')),
]

for r, (name, files_) in enumerate(groups):
    for c in range(4):
        img = Image.open(files_[c]).convert('RGB')
        axes[r, c].imshow(img)
        axes[r, c].axis('off')
        axes[r, c].set_title(name)

plt.tight_layout()
plt.show()

## 8. Create mutually exclusive train / val / test folders

We stratify by **source** so every split contains:
- real/FFHQ
- fake/StyleGAN2
- fake/diffusion

This is important for Run 2 because we want the validation and test sets to contain both fake-generation families.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

SEED = 42

sources = [
    ("real", "ffhq", 0, RAW_DIR / 'real'),
    ("fake", "stylegan2", 1, RAW_DIR / 'fake' / 'stylegan2'),
    ("fake", "diffusion", 1, RAW_DIR / 'fake' / 'diffusion'),
]

rows = []
for label_name, source, label, folder in sources:
    for p in image_files(folder):
        rows.append({
            "src": str(p),
            "label": label,
            "source": source
        })

df = pd.DataFrame(rows)
print("Total:", len(df))
print(df.groupby(["label","source"]).size())

strat_key = df["label"].astype(str) + "_" + df["source"]

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=strat_key, random_state=SEED
)

temp_key = temp_df["label"].astype(str) + "_" + temp_df["source"]

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_key, random_state=SEED
)

SPLITS = {"train": train_df, "val": val_df, "test": test_df}

# Remove old split folders so rerunning this cell cannot leave stale files.
FINAL_DIR = DATA_DIR
for split in SPLITS:
    for label_name in ["real", "fake"]:
        target = FINAL_DIR / split / label_name
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True, exist_ok=True)

for split, split_df in SPLITS.items():
    for _, row in split_df.iterrows():
        src = Path(row["src"])
        label_name = "real" if row["label"] == 0 else "fake"

        # Prefix source to avoid filename collisions.
        dst = FINAL_DIR / split / label_name / f"{row['source']}_{src.name}"
        shutil.copy2(src, dst)

    print(f"\n{split}: {len(split_df)}")
    print(split_df.groupby(["label","source"]).size())

print("\nFinal dataset created at:", FINAL_DIR)

## 9. Verify final directory structure

In [ ]:
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    for label in ["real", "fake"]:
        count = len(image_files(DATA_DIR / split / label))
        print(f"  {label}: {count}")

assert all((DATA_DIR / split / label).exists()
           for split in ["train","val","test"]
           for label in ["real","fake"])

print("\nDataset is ready for full_hybrid training.")

## 10. Save split manifests

These CSVs are for reproducibility/auditing only. The new training script uses the train/val/test directories.

In [ ]:
split_manifest_dir = DATA_DIR / "splits"
split_manifest_dir.mkdir(exist_ok=True)

for split, split_df in SPLITS.items():
    manifest = split_df.copy()
    manifest["split"] = split
    manifest.to_csv(split_manifest_dir / f"{split}.csv", index=False)

print("Saved manifests:", list(split_manifest_dir.glob("*.csv")))

# Done

Your Run 2 dataset is now:

```text
data/
├── train/
│   ├── real/
│   └── fake/
├── val/
│   ├── real/
│   └── fake/
└── test/
    ├── real/
    └── fake/
```

**Next:** run the updated training notebook.

Do not run the old `stage1` / `hybrid` cells for the main Run 2 experiment.